# Balance nuevo · La Grulla Blanca\nNotebook reutilizable para Google Colab. **No usa estadísticas de ver74.**\n\nCambiar pruebas = editar CONFIG/SCENARIOS. El motor permanece igual.

In [ ]:
!git clone -q -b experiment/combat-stat-contract-v0.1 https://github.com/Shein25/La-Grulla-Blanca.git /content/grulla\nimport sys\nsys.path.append('/content/grulla/experimentos/balance_nuevo')\nfrom sim_core import ActorStats, Technique, Dice\nfrom run_matrix import Scenario, run_matrix, export_csv\nimport pandas as pd

## 1. Configuración\nUse sólo valores `CANON`, `PROVISIONAL` o `LAB` explícitos. Si un valor de LianQi I todavía no fue cerrado, no lo copie de ver74.

In [ ]:
# EJEMPLO DE ESTRUCTURA, NO VALORES CANÓNICOS.\n# Reemplace None sólo cuando exista una decisión explícita de laboratorio/canon.\nLIANQI_I = {\n    'hp': None,\n    'qi': None,\n    'precision': 100.0,  # CANON: referencia normal del contrato nuevo\n    'evasion': None,\n    'defense': None,\n    'control': None,\n    'tenacity': None,\n}\n\ndef require_complete(d, name):\n    missing=[k for k,v in d.items() if v is None]\n    if missing:\n        raise ValueError(f'{name}: faltan valores nuevos explícitos: {missing}')\n\n# require_complete(LIANQI_I, 'LIANQI_I')  # activar al comenzar el benchmark serio

## 2. Técnica con daño variable\nLos `daño nominal` actuales son objetivos provisionales de balance, no daño fijo final. Declare aquí la distribución de dados candidata.

In [ ]:
# Ejemplo de uso una vez elegidos dados nuevos:\n# palma = Technique(name='Palma Ardiente', qi_cost=6, damage=Dice('...'))\n# enemigo = ActorStats(hp_max=..., qi_max=..., precision=..., evasion=..., defense=...)\n# jugador = ActorStats(hp_max=..., qi_max=..., precision=100, evasion=..., defense=...)

## 3. Matriz Monte Carlo\nAñada escenarios; el motor, métricas y exportación no se reescriben.

In [ ]:
scenarios = []\n# scenarios.append(Scenario(\n#     id='L1_FUEGO_COMMON_BASE', provenance='LAB',\n#     attacker=jugador, target=enemigo, technique=palma,\n#     iterations=200_000, seed=20260929))\n\nrows = run_matrix(scenarios) if scenarios else []\ndf = pd.DataFrame(rows)\ndf

In [ ]:
if rows:\n    export_csv(rows, '/content/resultados_balance.csv')\n    print('/content/resultados_balance.csv')

## Regla de interpretación\nUn resultado `LAB` puede orientar sensibilidad, pero no se convierte en canon hasta que los parámetros de entrada estén aprobados. Los resultados deben conservar semilla, iteraciones y procedencia.

## 4. Progresión por etapa\nCompare la política de control con variantes LAB. Las stats adicionales por ascenso nunca son canon automáticamente.

In [ ]:
from progression_lab import RESOURCES_ONLY, uniform_extra_policy, policy_signature\n\n# Ejemplo LAB; cambie valores sin tocar el motor:\npolicy_lab = uniform_extra_policy(\n    name='ejemplo_stats_ligeras',\n    precision_per_stage=2,\n    defense_per_stage=0.5,\n)\npd.DataFrame([\n    policy_signature(RESOURCES_ONLY, s) for s in ['LianQi_I','LianQi_II','LianQi_III','LianQi_IV']\n] + [\n    policy_signature(policy_lab, s) for s in ['LianQi_I','LianQi_II','LianQi_III','LianQi_IV']\n])

## 5. Equipo por etapa\nLas estadísticas legacy no se importan. El catálogo de auditoría sólo conserva identidad, slot y fuente.

In [ ]:
from equipment_lab import CURRENT_SOURCE_AUDIT, available_items\n\nfor stage in ['LianQi_I','LianQi_II','LianQi_III','LianQi_IV']:\n    print(stage, [x.item_id for x in available_items(CURRENT_SOURCE_AUDIT.values(), stage)])